<a href="https://colab.research.google.com/github/ddesiccantt/EECS498-DC/blob/main/A2_Q1_1_SVM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch

# 两个样本，每个样本有两个特征
X = torch.tensor([
    [1., 2.],
    [3., 4.]
])

# 每一列对应一个类别的权重
W = torch.tensor([
    [1., 0., -1.],
    [0., 1.,  1.]
])

scores = X @ W
print(scores)

predictions = scores.argmax(dim=1)
print(predictions)

scores = torch.tensor([1., 2., 1.])
y = 0  # 真实类别

correct_score = scores[y]

# 计算每个类别对应的 margin，并把负数变成 0
margins = (scores - correct_score + 1).clamp(min=0)

# 真实类别不参与求和
margins[y] = 0

loss = margins.sum()

print(margins)  # tensor([0., 2., 1.])
print(loss)     # tensor(3.)

tensor([[1., 2., 1.],
        [3., 4., 1.]])
tensor([1, 1])
tensor([0., 2., 1.])
tensor(3.)


In [ ]:
import torch

# ===== 1. 准备数据 =====

# X: [样本数 N, 特征数 D]
X = torch.tensor([
    [1., 2.],
    [3., 4.]
])

# W: [特征数 D, 类别数 C]
W = torch.tensor([
    [1., 0., -1.],
    [0., 1.,  1.]
])

# 每个样本的真实类别
y = torch.tensor([0, 2])


# ===== 2. 计算分数和预测类别 =====

scores = X @ W
predictions = scores.argmax(dim=1)

print("分类分数：")
print(scores)
# tensor([[1., 2., 1.],
#         [3., 4., 1.]])

print("预测类别：", predictions)
# tensor([1, 1])


# ===== 3. SVM loss：循环版 =====

def svm_loss_loop(scores, y):
    N = scores.shape[0]
    total_loss = scores.new_zeros(())

    for i in range(N):
        sample_scores = scores[i]
        correct_score = sample_scores[y[i]]

        margins = (sample_scores - correct_score + 1).clamp(min=0)
        margins[y[i]] = 0

        total_loss += margins.sum()

    return total_loss / N


# ===== 4. SVM loss：向量化版 =====

def svm_loss_vectorized(scores, y):
    N = scores.shape[0]
    rows = torch.arange(N, device=scores.device)

    # [N] → [N, 1]，方便广播
    correct_scores = scores[rows, y].unsqueeze(1)

    margins = (scores - correct_scores + 1).clamp(min=0)

    # 真实类别不参与求和
    margins[rows, y] = 0

    return margins.sum() / N


# ===== 5. 比较两种实现 =====

loss_loop = svm_loss_loop(scores, y)
loss_vectorized = svm_loss_vectorized(scores, y)

print("循环版 loss：", loss_loop.item())        # 5.0
print("向量化版 loss：", loss_vectorized.item())  # 5.0

print("结果是否一致：", torch.allclose(loss_loop, loss_vectorized))
# True

tensor(5.)


In [4]:
import torch

# ===== 1. 准备数据 =====

# X: [N, D]，两个样本，每个样本两个特征
X = torch.tensor([
    [1., 2.],
    [3., 4.]
])

# W: [D, C]，每一列对应一个类别
W = torch.tensor([
    [1., 0., -1.],
    [0., 1.,  1.]
])

# y: [N]，每个样本的真实类别
y = torch.tensor([0, 2])


# ===== 2. 计算分数和预测类别 =====

scores = X @ W
predictions = scores.argmax(dim=1)

print("分类分数：")
print(scores)
# tensor([[1., 2., 1.],
#         [3., 4., 1.]])

print("预测类别：", predictions)
# tensor([1, 1])


# ===== 3. SVM loss：循环版 =====

def svm_loss_loop(scores, y):
    N = scores.shape[0]
    total_loss = scores.new_zeros(())

    for i in range(N):
        sample_scores = scores[i]
        correct_score = sample_scores[y[i]]

        margins = (sample_scores - correct_score + 1).clamp(min=0)
        margins[y[i]] = 0

        total_loss += margins.sum()

    return total_loss / N


# ===== 4. SVM loss 和梯度：向量化版 =====

def svm_loss_and_gradient(X, W, y):
    N = X.shape[0]
    rows = torch.arange(N, device=X.device)

    # 计算分类分数：[N, C]
    scores = X @ W

    # 取出真实类别分数：[N] → [N, 1]
    correct_scores = scores[rows, y].unsqueeze(1)

    # 计算每个错误类别的损失
    margins = (scores - correct_scores + 1).clamp(min=0)
    margins[rows, y] = 0

    # 对样本取平均
    loss = margins.sum() / N

    # loss 对分数的梯度
    # 错误类别：margin > 0 时为 1，否则为 0
    dscores = (margins > 0).to(scores.dtype)

    # 真实类别：正 margin 的错误类别数量取负数
    counts = dscores.sum(dim=1)
    dscores[rows, y] = -counts

    # loss 取了平均，梯度也要取平均
    dscores /= N

    # loss 对权重 W 的梯度：[D, N] @ [N, C] → [D, C]
    dW = X.T @ dscores

    return loss, dW


# ===== 5. 运行并比较 =====

loss_loop = svm_loss_loop(scores, y)
loss_vectorized, dW = svm_loss_and_gradient(X, W, y)

print("循环版 loss：", loss_loop.item())
# 5.0

print("向量化版 loss：", loss_vectorized.item())
# 5.0

print("loss 是否一致：", torch.allclose(loss_loop, loss_vectorized))
# True

print("W 的梯度：")
print(dW)
# tensor([[ 0.5000,  2.0000, -2.5000],
#         [ 0.0000,  3.0000, -3.0000]])

learning_rate = 0.1

# 1. 用当前权重计算 loss 和梯度
loss_before, dW = svm_loss_and_gradient(X, W, y)

# 2. 更新所有权重
W = W - learning_rate * dW

# 3. 用新权重重新计算 loss
loss_after, _ = svm_loss_and_gradient(X, W, y)

print("更新后的 W：")
print(W)
# tensor([[ 0.9500, -0.2000, -0.7500],
#         [ 0.0000,  0.7000,  1.3000]])

print("更新前 loss：", loss_before.item())  # 5.0
print("更新后 loss：", loss_after.item())   # 约 2.225

# 重新设置初始权重
W = torch.tensor([
    [1., 0., -1.],
    [0., 1.,  1.]
])

learning_rate = 0.1

for step in range(20):
    # 计算当前 loss 和权重梯度
    loss, dW = svm_loss_and_gradient(X, W, y)

    print(f"第 {step + 1} 次更新前，loss = {loss.item():.4f}")

    # 更新权重
    W = W - learning_rate * dW

# 查看训练后的结果
final_loss, _ = svm_loss_and_gradient(X, W, y)
predictions = (X @ W).argmax(dim=1)

print("最终 loss：", final_loss.item())
print("预测类别：", predictions)
print("真实类别：", y)

分类分数：
tensor([[1., 2., 1.],
        [3., 4., 1.]])
预测类别： tensor([1, 1])
循环版 loss： 5.0
向量化版 loss： 5.0
loss 是否一致： True
W 的梯度：
tensor([[ 0.5000,  2.0000, -2.5000],
        [ 0.0000,  3.0000, -3.0000]])
更新后的 W：
tensor([[ 0.9500, -0.2000, -0.7500],
        [ 0.0000,  0.7000,  1.3000]])
更新前 loss： 5.0
更新后 loss： 2.1500000953674316
第 1 次更新前，loss = 5.0000
第 2 次更新前，loss = 2.1500
第 3 次更新前，loss = 1.6500
第 4 次更新前，loss = 1.2500
第 5 次更新前，loss = 1.3500
第 6 次更新前，loss = 1.3000
第 7 次更新前，loss = 1.2500
第 8 次更新前，loss = 1.2000
第 9 次更新前，loss = 1.3000
第 10 次更新前，loss = 1.1000
第 11 次更新前，loss = 1.3500
第 12 次更新前，loss = 1.1000
第 13 次更新前，loss = 1.3000
第 14 次更新前，loss = 1.1500
第 15 次更新前，loss = 1.2000
第 16 次更新前，loss = 1.2000
第 17 次更新前，loss = 1.1000
第 18 次更新前，loss = 1.2500
第 19 次更新前，loss = 1.0000
第 20 次更新前，loss = 1.3000
最终 loss： 1.0499998331069946
预测类别： tensor([2, 2])
真实类别： tensor([0, 2])
